In [15]:
import asyncio
from playwright.async_api import async_playwright
import pandas as pd
import re

async def scrape_page(context, url, idx, sem):
    async with sem:
        page = await context.new_page()
        # Blocking ONLY images to keep the UI structure 100% intact
        await page.route("**/*.{png,jpg,jpeg,svg}", lambda route: route.abort())
        
        print(f"🚀 Scanning Page {idx}...")
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            
            # Dynamic Wait & Scroll
            # We scroll multiple times to ensure the '18' or '24' card threshold is met
            for _ in range(15):
                await page.mouse.wheel(0, 800)
                await asyncio.sleep(0.5)
            
            # Final buffer for JS rendering
            await asyncio.sleep(4) 
            
            # Capture ALL card containers
            cards = await page.query_selector_all('div[data-testid="card-container"]')
            
            page_data = []
            for card in cards:
                text = await card.inner_text()
                # Extremely permissive regex to find ANY number near ₺ or TL
                price_matches = re.findall(r'(\d[\d\.,\s]*)\s?(?:₺|TL)', text)
                
                # Clean the strings into integers
                clean_prices = [int(re.sub(r'[^\d]', '', p)) for p in price_matches if len(re.sub(r'[^\d]', '', p)) > 0]
                
                # Filter out small numbers (like 1 guest, 4 adults) but keep the price
                prices = [p for p in clean_prices if p > 500]
                
                if prices:
                    # Airbnb displays total as the last price in the string
                    final_gross = prices[-1]
                    page_data.append({
                        "nightly_net": round((final_gross / 2) * 0.85, 2),
                        "gross_total": final_gross
                    })
                else:
                    # Even if no price is found, we add a placeholder to keep the count correct
                    # This ensures the card is 'counted' in your 103 total
                    page_data.append({
                        "nightly_net": 0, 
                        "gross_total": 0
                    })
            
            print(f"✅ Page {idx}: Scanned {len(cards)} cards.")
            await page.close()
            return page_data
        except Exception as e:
            print(f"❌ Error on Page {idx}: {str(e)[:50]}")
            await page.close()
            return []

async def run_market_scan(urls):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            viewport={'width': 1280, 'height': 1200},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
        )
        sem = asyncio.Semaphore(2)
        tasks = [scrape_page(context, url, i+1, sem) for i, url in enumerate(urls)]
        results = await asyncio.gather(*tasks)
        await browser.close()
        return [item for sublist in results for item in sublist]

In [19]:
# --- CONFIGURATION ---
target_urls = [
    "https://www.airbnb.com.tr/s/Beyo%C4%9Flu--%C4%B0stanbul/homes?place_id=ChIJPTZw2Fq3yhQRsO8zBbUFvCU&refinement_paths%5B%5D=%2Fhomes&flexible_trip_dates%5B%5D=april&flexible_trip_lengths%5B%5D=weekend_trip&date_picker_type=calendar&adults=4&guests=4&query=Beyo%C4%9Flu%2C%20%C4%B0stanbul&monthly_start_date=2026-04-01&monthly_length=3&monthly_end_date=2026-07-01&price_filter_input_type=2&price_filter_num_nights=2&channel=EXPLORE&checkin=2026-04-10&checkout=2026-04-12&source=structured_search_input_header&ne_lat=41.03504736544088&ne_lng=28.982286824298598&sw_lat=41.023028518736396&sw_lng=28.969879294524276&zoom=15.947584163115053&zoom_level=15&search_by_map=true&selected_filter_order%5B%5D=room_types%3AEntire%20home%2Fapt&selected_filter_order%5B%5D=l2_property_type_ids%3A3&selected_filter_order%5B%5D=amenities%3A8&selected_filter_order%5B%5D=amenities%3A30&selected_filter_order%5B%5D=price_min%3A5000&selected_filter_order%5B%5D=price_max%3A12000&update_selected_filters=true&room_types%5B%5D=Entire%20home%2Fapt&l2_property_type_ids%5B%5D=3&amenities%5B%5D=8&amenities%5B%5D=30&search_mode=regular_search&disable_auto_translation=true&price_min=5000&search_type=filter_change&price_max=12000",
    "https://www.airbnb.com.tr/s/Beyo%C4%9Flu--%C4%B0stanbul/homes?place_id=ChIJPTZw2Fq3yhQRsO8zBbUFvCU&refinement_paths%5B%5D=%2Fhomes&flexible_trip_dates%5B%5D=april&flexible_trip_lengths%5B%5D=weekend_trip&date_picker_type=calendar&adults=4&guests=4&query=Beyo%C4%9Flu%2C%20%C4%B0stanbul&monthly_start_date=2026-04-01&monthly_length=3&monthly_end_date=2026-07-01&price_filter_input_type=2&price_filter_num_nights=2&channel=EXPLORE&checkin=2026-04-10&checkout=2026-04-12&source=structured_search_input_header&ne_lat=41.03504736544088&ne_lng=28.982286824298598&sw_lat=41.023028518736396&sw_lng=28.969879294524276&zoom=15.947584163115053&zoom_level=15&search_by_map=true&selected_filter_order%5B%5D=room_types%3AEntire%20home%2Fapt&selected_filter_order%5B%5D=l2_property_type_ids%3A3&selected_filter_order%5B%5D=amenities%3A8&selected_filter_order%5B%5D=amenities%3A30&selected_filter_order%5B%5D=price_min%3A5000&selected_filter_order%5B%5D=price_max%3A12000&update_selected_filters=true&room_types%5B%5D=Entire%20home%2Fapt&l2_property_type_ids%5B%5D=3&amenities%5B%5D=8&amenities%5B%5D=30&price_min=5000&price_max=12000&disable_auto_translation=true&federated_search_session_id=723e4e5a-5b93-43f6-b61a-575c6912d0d1&pagination_search=true&cursor=eyJzZWN0aW9uX29mZnNldCI6MCwiaXRlbXNfb2Zmc2V0IjoxOCwidmVyc2lvbiI6MX0%3D",
    "https://www.airbnb.com.tr/s/Beyo%C4%9Flu--%C4%B0stanbul/homes?place_id=ChIJPTZw2Fq3yhQRsO8zBbUFvCU&refinement_paths%5B%5D=%2Fhomes&flexible_trip_dates%5B%5D=april&flexible_trip_lengths%5B%5D=weekend_trip&date_picker_type=calendar&adults=4&guests=4&query=Beyo%C4%9Flu%2C%20%C4%B0stanbul&monthly_start_date=2026-04-01&monthly_length=3&monthly_end_date=2026-07-01&price_filter_input_type=2&price_filter_num_nights=2&channel=EXPLORE&checkin=2026-04-10&checkout=2026-04-12&source=structured_search_input_header&ne_lat=41.03504736544088&ne_lng=28.982286824298598&sw_lat=41.023028518736396&sw_lng=28.969879294524276&zoom=15.947584163115053&zoom_level=15&search_by_map=true&selected_filter_order%5B%5D=room_types%3AEntire%20home%2Fapt&selected_filter_order%5B%5D=l2_property_type_ids%3A3&selected_filter_order%5B%5D=amenities%3A8&selected_filter_order%5B%5D=amenities%3A30&selected_filter_order%5B%5D=price_min%3A5000&selected_filter_order%5B%5D=price_max%3A12000&update_selected_filters=true&room_types%5B%5D=Entire%20home%2Fapt&l2_property_type_ids%5B%5D=3&amenities%5B%5D=8&amenities%5B%5D=30&price_min=5000&price_max=12000&pagination_search=true&disable_auto_translation=true&federated_search_session_id=723e4e5a-5b93-43f6-b61a-575c6912d0d1&cursor=eyJzZWN0aW9u_zZWN0aW9uX29mZnNldCI6MCwiaXRlbXNfb2Zmc2V0IjozNiwidmVyc2lvbiI6MX0%3D",
    "https://www.airbnb.com.tr/s/Beyo%C4%9Flu--%C4%B0stanbul/homes?place_id=ChIJPTZw2Fq3yhQRsO8zBbUFvCU&refinement_paths%5B%5D=%2Fhomes&flexible_trip_dates%5B%5D=april&flexible_trip_lengths%5B%5D=weekend_trip&date_picker_type=calendar&adults=4&guests=4&query=Beyo%C4%9Flu%2C%20%C4%B0stanbul&monthly_start_date=2026-04-01&monthly_length=3&monthly_end_date=2026-07-01&price_filter_input_type=2&price_filter_num_nights=2&channel=EXPLORE&checkin=2026-04-10&checkout=2026-04-12&source=structured_search_input_header&ne_lat=41.03504736544088&ne_lng=28.982286824298598&sw_lat=41.023028518736396&sw_lng=28.969879294524276&zoom=15.947584163115053&zoom_level=15&search_by_map=true&selected_filter_order%5B%5D=room_types%3AEntire%20home%2Fapt&selected_filter_order%5B%5D=l2_property_type_ids%3A3&selected_filter_order%5B%5D=amenities%3A8&selected_filter_order%5B%5D=amenities%3A30&selected_filter_order%5B%5D=price_min%3A5000&selected_filter_order%5B%5D=price_max%3A12000&update_selected_filters=true&room_types%5B%5D=Entire%20home%2Fapt&l2_property_type_ids%5B%5D=3&amenities%5B%5D=8&amenities%5B%5D=30&price_min=5000&price_max=12000&pagination_search=true&disable_auto_translation=true&federated_search_session_id=723e4e5a-5b93-43f6-b61a-575c6912d0d1&cursor=eyJzZWN0aW9uX29mZnNldCI6MCwiaXRlbXNfb2Zmc2V0Ijo1NCwidmVyc2lvbiI6MX0%3D",
    "https://www.airbnb.com.tr/s/Beyo%C4%9Flu--%C4%B0stanbul/homes?place_id=ChIJPTZw2Fq3yhQRsO8zBbUFvCU&refinement_paths%5B%5D=%2Fhomes&flexible_trip_dates%5B%5D=april&flexible_trip_lengths%5B%5D=weekend_trip&date_picker_type=calendar&adults=4&guests=4&query=Beyo%C4%9Flu%2C%20%C4%B0stanbul&monthly_start_date=2026-04-01&monthly_length=3&monthly_end_date=2026-07-01&price_filter_input_type=2&price_filter_num_nights=2&channel=EXPLORE&checkin=2026-04-10&checkout=2026-04-12&source=structured_search_input_header&ne_lat=41.03504736544088&ne_lng=28.982286824298598&sw_lat=41.023028518736396&sw_lng=28.969879294524276&zoom=15.947584163115053&zoom_level=15&search_by_map=true&selected_filter_order%5B%5D=room_types%3AEntire%20home%2Fapt&selected_filter_order%5B%5D=l2_property_type_ids%3A3&selected_filter_order%5B%5D=amenities%3A8&selected_filter_order%5B%5D=amenities%3A30&selected_filter_order%5B%5D=price_min%3A5000&selected_filter_order%5B%5D=price_max%3A12000&update_selected_filters=true&room_types%5B%5D=Entire%20home%2Fapt&l2_property_type_ids%5B%5D=3&amenities%5B%5D=8&amenities%5B%5D=30&price_min=5000&price_max=12000&pagination_search=true&disable_auto_translation=true&federated_search_session_id=723e4e5a-5b93-43f6-b61a-575c6912d0d1&cursor=eyJzZWN0aW9uX29mZnNldCI6MCwiaXRlbXNfb2Zmc2V0Ijo3MiwidmVyc2lvbiI6MX0%3D"
]

# Run the Scraper
all_market_data = await run_market_scan(target_urls)

# Process Results
df_raw = pd.DataFrame(all_market_data)
df_unique = df_raw

if not df_unique.empty:
    market_capacity = 220
    available_listings = len(df_unique)
    occupancy_rate = (market_capacity - available_listings) / market_capacity
    median_net = df_unique['nightly_net'].median()
    
    # Dynamic Multipliers
    multiplier = 1.25 if occupancy_rate > 0.80 else 1.10 if occupancy_rate > 0.60 else 1.00
    target_net = median_net * multiplier
    suggested_gross = (target_net * 2) / 0.85
    
    print("\n" + "="*50)
    print(f"AIRBNB DYNAMIC PRICING: BEYOĞLU")
    print("="*50)
    print(f"Total Scanned (Raw)   : {len(df_raw)}")
    print(f"Unique Available      : {available_listings} / {market_capacity}")
    print(f"Occupancy Rate        : {occupancy_rate:.1%}")
    print(f"Competitor Median Net : {median_net:,.0f} ₺")
    print("-" * 50)
    print(f"YOUR SUGGESTED NET    : {target_net:,.0f} ₺")
    print(f"LISTING TOTAL (2 NTS) : {round(suggested_gross, -1):,.0f} ₺")
    print("="*50)
else:
    print("No data collected. Check your internet or URLs.")

🚀 Scanning Page 1...
🚀 Scanning Page 2...
✅ Page 1: Scanned 24 cards.
🚀 Scanning Page 3...
✅ Page 2: Scanned 24 cards.
🚀 Scanning Page 4...
✅ Page 3: Scanned 24 cards.
🚀 Scanning Page 5...
✅ Page 4: Scanned 18 cards.
✅ Page 5: Scanned 18 cards.

AIRBNB DYNAMIC PRICING: BEYOĞLU
Total Scanned (Raw)   : 108
Unique Available      : 108 / 220
Occupancy Rate        : 50.9%
Competitor Median Net : 4,801 ₺
--------------------------------------------------
YOUR SUGGESTED NET    : 4,801 ₺
LISTING TOTAL (2 NTS) : 11,300 ₺
